# Phase 4: Model Evaluation and Explainability
This notebook evaluates the best performing model using business metrics and computes SHAP/permutation explainability.


## Step 1: Evaluate Best Model on Test Set
We evaluate the tuned Gradient Boosting classifier on the held-out test set.


In [1]:
import pandas as pd
import numpy as np
from src.data_pipeline.preprocessing import split_data, scale_features
from src.models.advanced import train_gradient_boosting
from src.evaluation.metrics import compute_classification_report, compute_threshold_analysis, compute_cost_sensitive_metrics
from src.evaluation.explainability import compute_permutation_importance, compute_shap_summary, get_top_failure_drivers

df = pd.read_csv('data/features/engineered_features.csv')
X = df.drop(columns=['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'])
y = df['Machine failure']

X_train, X_val, X_test, y_train, y_val, y_test = split_data(
    pd.concat([X, y], axis=1), 'Machine failure'
)
X_train_s, X_val_s, X_test_s, scaler = scale_features(X_train, X_val, X_test)

best_model = train_gradient_boosting(X_train_s, y_train)
y_pred = best_model.predict(X_test_s)
y_proba = best_model.predict_proba(X_test_s)[:, 1]

report = compute_classification_report(y_test, y_pred, y_proba)
print('Classification Report on Test Set:')
for k, v in report.items():
    print(f'{k}: {v}')


C:\Users\hp\Downloads\predictive_maintenance\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Classification Report on Test Set:
accuracy: 0.9825
precision: 0.8468
recall: 0.92
f1_macro: 0.8796
f1_weighted: 0.9833
roc_auc: 0.9667
average_precision: 0.8882
matthews_corrcoef: 0.7633
confusion_matrix: [[1908, 25], [10, 58]]


## Step 2: Cost-Sensitive Threshold Analysis
We find the optimal threshold that minimizes business maintenance costs.


In [2]:
thresh_df = compute_threshold_analysis(y_test, y_proba)
cost_matrix = {'TP': -500, 'TN': 0, 'FP': 200, 'FN': 5000}

costs = []
for idx, row in thresh_df.iterrows():
    t = row['threshold']
    y_pred_t = (y_proba >= t).astype(int)
    cost_res = compute_cost_sensitive_metrics(y_test, y_pred_t, cost_matrix)
    costs.append(cost_res['total_cost'])

thresh_df['total_cost'] = costs
best_threshold_row = thresh_df.sort_values('total_cost', ascending=True).iloc[0]
print('Optimal threshold row:')
print(best_threshold_row)


Optimal threshold row:
threshold                 0.2000
precision                 0.4599
recall                    0.9265
f1                        0.6146
predicted_positives     137.0000
total_cost             8300.0000
Name: 2, dtype: float64


## Step 3: Permutation Importance & SHAP Explainability
We compute model explainability using permutation importance and SHAP values.


In [3]:
X_val_df = pd.DataFrame(X_val_s, columns=X.columns)
perm_df = compute_permutation_importance(best_model, X_val_df, y_val)
print('Permutation Importance:')
print(perm_df.head(5))

print('\nComputing SHAP values...')
shap_res = compute_shap_summary(best_model, X_val_df, model_type='tree')
top_drivers = get_top_failure_drivers(shap_res, n_top=5)
print('Top 5 Failure Drivers by SHAP:')
print(top_drivers)


C:\Users\hp\Downloads\predictive_maintenance\venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but GradientBoostingClassifier was fitted without feature names
  warnings.warn(


Permutation Importance:
                        feature  importance_mean  importance_std
0                  temp_delta_K           0.0180        0.002049
1        Rotational speed [rpm]           0.0163        0.002900
2  tool_wear_torque_interaction           0.0104        0.002107
3                   power_watts           0.0092        0.002358
4                  strain_index           0.0025        0.001118

Computing SHAP values...


Top 5 Failure Drivers by SHAP:
['power_watts', 'Rotational speed [rpm]', 'Tool wear [min]', 'Torque [Nm]', 'temp_delta_K']
